# Spiking Neural Networks with sinabs — DVS Gesture
### A hands-on tutorial: from neuron to trained model

**Prerequisites:** Python, basic PyTorch. 

**Dataset:** IBM DVS Gesture — 11 hand gesture classes recorded with a Dynamic Vision Sensor (DVS) camera.

**Why this dataset is a better motivating example than N-MNIST:**  
A hand gesture is defined entirely by motion over time. A single frame of a gesture is nearly unrecognisable — you need the sequence. This makes DVS Gesture a genuinely temporal task, where the SNN's ability to integrate information over time is not just a design choice but a necessity.

## Before you start

Activate the `sinabs_tutorial` conda environment as your kernel, and make sure you have downloaded the DVS Gesture dataset. See [README.md](README.md) for setup and dataset download instructions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import tonic
import tonic.transforms as transforms
import sinabs
import sinabs.layers as sl
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
print(f"sinabs version: {sinabs.__version__}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. Why Spikes?

A standard artificial neuron outputs a **continuous value** — a floating-point number passed through ReLU or sigmoid. Biological neurons communicate very differently: they either fire an electrical pulse (a **spike**, value = 1) or stay silent (value = 0).

```
Standard neuron:   output = ReLU(w·x + b)     ← real number, every step
Spiking neuron:    output = 0 or 1             ← binary event, only when threshold crossed
```

**Why bother?**
1. **Energy efficiency** — a spike is transmitted only when something happens. Silent neurons cost almost nothing on neuromorphic hardware.
2. **Event-driven sensors** — DVS cameras produce exactly this format: sparse binary events in time. SNNs process them natively.
3. **Temporal dynamics** — a spiking neuron accumulates charge over time before firing. For a gesture that unfolds over hundreds of milliseconds, this memory is essential.

The key challenge: if the neuron output is always 0 or 1, how do we train with backpropagation? We will answer this in Section 5.

---
## 2. Event-Based Data: DVS Gesture

**IBM DVS Gesture** contains 1,077 training and 264 test recordings of 11 hand gestures performed under 3 different lighting conditions, captured with a DVS event camera at 128×128 pixel resolution.

### The 11 gesture classes

| Index | Gesture |
|---|---|
| 0 | Hand clapping |
| 1 | Right hand wave |
| 2 | Left hand wave |
| 3 | Right arm clockwise |
| 4 | Right arm counter-clockwise |
| 5 | Left arm clockwise |
| 6 | Left arm counter-clockwise |
| 7 | Arm roll |
| 8 | Air drums |
| 9 | Air guitar |
| 10 | Other gestures |

Each recording is a stream of events. We accumulate them into fixed time windows:

```
static image:        (C, H, W)            e.g. (3, 128, 128)
DVS Gesture sample:  (T, C, H, W)         e.g. (20, 2, 128, 128)
                      ↑                          ↑
                 time slices           ON/OFF polarity channels
```

Unlike N-MNIST, a single frame here carries almost no class information — the gesture identity is in the motion pattern across all T timesteps.

In [ ]:
NUM_TIMESTEPS = 20
SENSOR_SIZE = tonic.datasets.DVSGesture.sensor_size  # (128, 128, 2)

GESTURE_CLASSES = [
    "Hand clapping", "Right hand wave", "Left hand wave",
    "Right arm CW", "Right arm CCW", "Left arm CW",
    "Left arm CCW", "Arm roll", "Air drums", "Air guitar", "Other"
]

frame_transform = transforms.Compose([
    transforms.Denoise(filter_time=10_000),
    transforms.ToFrame(sensor_size=SENSOR_SIZE, n_time_bins=NUM_TIMESTEPS),
])

dataset_train_raw = tonic.datasets.DVSGesture(save_to="./data", train=True,  transform=frame_transform)
dataset_test_raw  = tonic.datasets.DVSGesture(save_to="./data", train=False, transform=frame_transform)

# Cache preprocessed frames — slow on first run, fast every run after
dataset_train = tonic.DiskCachedDataset(dataset_train_raw, cache_path="./data/cache/dvs_gesture_train")
dataset_test  = tonic.DiskCachedDataset(dataset_test_raw,  cache_path="./data/cache/dvs_gesture_test")

print(f"Training samples: {len(dataset_train)}")
print(f"Test samples:     {len(dataset_test)}")

sample, label = dataset_train[0]
print(f"\nOne sample shape: {sample.shape}   → (T, C, H, W)")
print(f"Label: {label}  ({GESTURE_CLASSES[label]})")

In [ ]:
# Find a "Hand clapping" sample — clearest motion pattern across timesteps
sample_np, vis_label = next((s, l) for s, l in dataset_train if l == 0)

t_indices = np.linspace(0, NUM_TIMESTEPS - 1, 10, dtype=int)

fig, axes = plt.subplots(2, 10, figsize=(26, 9))
fig.suptitle(
    f'DVS Gesture sample — "{GESTURE_CLASSES[vis_label]}" (label {vis_label})\n'
    f'Top: ON events  |  Bottom: OFF events  |  Each column = one timestep',
    fontsize=13, fontweight='bold'
)

for col, t in enumerate(t_indices):
    for row, (cmap, vmax) in enumerate(zip(['Reds', 'Blues'], [sample_np[t, 0].max() + 1, sample_np[t, 1].max() + 1])):
        ax = axes[row, col]
        ax.imshow(sample_np[t, row], cmap=cmap, vmin=0, vmax=max(vmax, 1))
        ax.set_title(f't={t}', fontsize=9)
        ax.axis('off')

axes[0, 0].set_ylabel('ON', fontsize=11, rotation=0, labelpad=22)
axes[1, 0].set_ylabel('OFF', fontsize=11, rotation=0, labelpad=22)
plt.tight_layout()
plt.show()

print("Notice: any single frame is nearly unrecognisable. The motion across frames defines the gesture.")

In [ ]:
# Show one sample from each gesture class
fig, axes = plt.subplots(2, 11, figsize=(28, 9))
fig.suptitle('One mid-sequence frame per gesture class (ON top, OFF bottom)', fontsize=13, fontweight='bold')

seen = {}
idx = 0
while len(seen) < 11 and idx < len(dataset_train):
    s, l = dataset_train[idx]
    if l not in seen:
        seen[l] = s
    idx += 1

for col in range(11):
    s = seen[col]
    t_mid = NUM_TIMESTEPS // 2
    for row, cmap in enumerate(['Reds', 'Blues']):
        ax = axes[row, col]
        ax.imshow(s[t_mid, row], cmap=cmap, vmin=0)
        ax.axis('off')
        if row == 0:
            ax.set_title(GESTURE_CLASSES[col], fontsize=8, wrap=True)

plt.tight_layout()
plt.show()

---
## 3. The Integrate-and-Fire (IAF) Neuron

The IAF neuron is the simplest spiking neuron model. It has one internal variable: the **membrane voltage** `Vmem`.

At each timestep:

```
① Integrate:   Vmem  ←  Vmem + input
② Fire:        spike =  floor(Vmem / threshold)   (0 if below; 1, 2, … if above)
③ Reset:       Vmem  ←  Vmem − spike × threshold
               (subtracts once per spike — leftover charge is preserved)
```

The crucial point: `Vmem` carries over from one timestep to the next. For a gesture, weak motion cues at t=0,1,2 can accumulate and eventually cause a spike at t=5. The neuron integrates evidence over time — exactly what is needed to recognise a gesture that unfolds over 20 timesteps.

**sinabs default (`MultiSpike`):** a neuron that accumulates Vmem = 2.3 with threshold = 1.0 fires twice in one step and resets to 0.3. With normalised inputs, Vmem rarely exceeds 2 × threshold, so you will almost always see at most one spike per step.

In [ ]:
def iaf_neuron_sim(inputs, threshold=1.0):
    """Simulate a single IAF neuron matching sinabs defaults:
    MultiSpike (floor(Vmem/threshold) spikes per step) + MembraneSubtract reset."""
    T = len(inputs)
    vmem = np.zeros(T + 1)
    spikes = np.zeros(T)
    for t in range(T):
        vmem[t + 1] = vmem[t] + inputs[t]                  # ① integrate
        if vmem[t + 1] >= threshold:                        # ② fire
            spikes[t] = np.floor(vmem[t + 1] / threshold)  # MultiSpike: fire as many times as possible
            vmem[t + 1] -= spikes[t] * threshold            # ③ reset: subtract once per spike
    return vmem[1:], spikes


# Simulate two neurons: one responding to fast early motion, one to slow sustained motion
T = 20
inputs_burst  = np.array([0.5, 0.6, 0.4, 0.1, 0.05, 0.05, 0.1, 0.05, 0.05,
                           0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])
inputs_steady = np.full(T, 0.12)

vmem_burst,  spikes_burst  = iaf_neuron_sim(inputs_burst)
vmem_steady, spikes_steady = iaf_neuron_sim(inputs_steady)
timesteps = np.arange(T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey='row')
fig.suptitle('IAF Neuron over 20 timesteps — burst input vs steady input', fontsize=13, fontweight='bold')
threshold = 1.0
scenarios = [
    ('Burst input (fast early motion)',    inputs_burst,  vmem_burst,  spikes_burst),
    ('Steady input (slow sustained motion)', inputs_steady, vmem_steady, spikes_steady),
]
for col, (title, inp, vmem, spikes) in enumerate(scenarios):
    ax0, ax1 = axes[0, col], axes[1, col]
    ax0.bar(timesteps, inp, color='steelblue', alpha=0.8, width=0.7)
    ax0.set_title(title, fontsize=11); ax0.set_ylabel('Input current', fontsize=10)
    ax0.set_ylim(0, 0.75); ax0.set_xticks(timesteps[::2]); ax0.grid(axis='y', alpha=0.3)

    spike_times = np.where(spikes > 0)[0]
    vmem_display = vmem.copy()
    for t in spike_times:
        vmem_display[t] = vmem[t] + spikes[t] * threshold

    ax1.plot(timesteps, vmem_display, color='darkorange', linewidth=2, marker='o', markersize=4, label='Vmem')
    ax1.axhline(threshold, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {threshold}')
    for t in spike_times:
        ax1.annotate('', xy=(t, vmem[t]), xytext=(t, vmem_display[t]),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, connectionstyle='arc3,rad=0.4'))
        lbl = 'spike' if spikes[t] == 1 else f'{int(spikes[t])}×'
        ax1.annotate('', xy=(t, vmem_display[t] + 0.25), xytext=(t, vmem_display[t]),
                     arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax1.text(t, vmem_display[t] + 0.32, lbl, ha='center', fontsize=8, color='red', fontweight='bold')
    ax1.set_ylabel('Vmem', fontsize=10); ax1.set_xlabel('Timestep', fontsize=10)
    ax1.set_xticks(timesteps[::2]); ax1.set_ylim(-0.1, 1.9); ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)
    ax1.text(0.98, 0.05, f'Total spikes: {int(spikes.sum())}', transform=ax1.transAxes,
             ha='right', fontsize=10, color='red')

plt.tight_layout()
plt.show()

**What to notice:**
- The burst neuron fires quickly from concentrated early input, then stays quiet
- The steady neuron fires more regularly — it "counts" the sustained motion
- Both behaviours are useful for gesture recognition: some gestures are quick (clap), others are slow arcs (arm roll)

---
## 4. From Single Neuron to a Network: sinabs and IAFSqueeze

### The batching challenge

Standard `nn.Conv2d` accepts 4D tensors `(Batch, C, H, W)` — no concept of time. Our data has shape `(Batch, T, C, H, W)`. sinabs solves this with the **squeeze trick**:

```
Input:          (B,  T,  C,   H,   W)     e.g. (16, 20, 2, 128, 128)
                  ↓  squeeze: B × T
To Conv2d:      (320, 2, 128, 128)         ← treated as a batch of 320 images
                  ↓  Conv2d (stateless)
                (320, N, 128, 128)
                  ↓  IAFSqueeze (knows B=16, T=20)
                      → loops over 20 timesteps internally
                      → maintains one Vmem buffer per real sample
                (320, N, 128, 128)         ← binary spikes
```

### Architecture

DVS Gesture uses 128×128 pixels — much larger than N-MNIST's 34×34. We need more aggressive spatial pooling to bring the feature maps to a manageable size before the classifier.

```
(B*T, 2, 128, 128)
    → Conv2d(2→16)   + IAFSqueeze → AvgPool2d(4) → (B*T, 16, 32, 32)
    → Conv2d(16→32)  + IAFSqueeze → AvgPool2d(4) → (B*T, 32,  8,  8)
    → Conv2d(32→64)  + IAFSqueeze → AvgPool2d(2) → (B*T, 64,  4,  4)
    → Flatten
    → Linear(64×4×4 → 11)        + IAFSqueeze   ← 11 gesture classes
```

In [ ]:
BATCH_SIZE = 16  # DVS Gesture has fewer samples; smaller batch works well

x_demo = torch.zeros(BATCH_SIZE, NUM_TIMESTEPS, 2, 128, 128)
print(f"Original input:    {tuple(x_demo.shape)}   (B, T, C, H, W)")

x_squeezed = x_demo.reshape(BATCH_SIZE * NUM_TIMESTEPS, 2, 128, 128)
print(f"After squeeze:     {tuple(x_squeezed.shape)}   (B×T, C, H, W)")

conv = nn.Conv2d(2, 16, kernel_size=3, padding=1)
print(f"After Conv2d:      {tuple(conv(x_squeezed).shape)}   (B×T, filters, H, W)")

In [ ]:
def build_snn(batch_size, num_timesteps):
    return nn.Sequential(
        # Block 1: 128 → 32
        nn.Conv2d(2, 16, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(4),

        # Block 2: 32 → 8
        nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(4),

        # Block 3: 8 → 4
        nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(2),

        # Classifier
        nn.Flatten(),
        nn.Linear(64 * 4 * 4, 11, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
    )


model = build_snn(BATCH_SIZE, NUM_TIMESTEPS)
print(model)

dummy = torch.zeros(BATCH_SIZE * NUM_TIMESTEPS, 2, 128, 128)
with torch.no_grad():
    out = model(dummy)
print(f"\nOutput shape: {tuple(out.shape)}   (B×T, num_classes)")
print(f"Parameters:   {sum(p.numel() for p in model.parameters()):,}")

---
## 5. BPTT and the Surrogate Gradient

### The problem: dead gradients

Training requires gradients. The spike function is a step:

```
spike = 1  if  Vmem ≥ threshold,  else 0

d(spike)/d(Vmem) = 0   almost everywhere
```

Zero gradient → weights never update → network cannot learn.

### The fix: surrogate gradient

During the **backward pass only**, replace the zero slope with a smooth approximation peaked at the threshold. The forward pass still produces true binary spikes.

```
Forward:   spike = step(Vmem - threshold)        ← true binary (0 or 1)
Backward:  d(spike)/d(Vmem) ≈ smooth_fn(Vmem)   ← surrogate (non-zero near threshold)
```

In [ ]:
vmem_vals = np.linspace(-1.5, 2.5, 500)
threshold = 1.0
v = vmem_vals - threshold
step = (vmem_vals >= threshold).astype(float)

beta, sigma = 4.0, 0.4
h, h2, mu2, s1, s2 = 0.5, 0.25, 0.5, 0.3, 0.3
sg_exp = np.exp(-beta * np.abs(v))
sg_gau = np.exp(-0.5 * (v / sigma) ** 2); sg_gau /= sg_gau.max()
sg_mul = (h  * np.exp(-0.5 * (v / s1) ** 2)
          - h2 * np.exp(-0.5 * ((v + mu2) / s2) ** 2)
          - h2 * np.exp(-0.5 * ((v - mu2) / s2) ** 2))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Surrogate gradients: forward spike (dashed black) vs backward gradient (colour)',
             fontsize=12, fontweight='bold')
for ax, name, sg, color in zip(axes,
        ['SingleExponential\n(default in sinabs)', 'Gaussian', 'MultiGaussian\n(negative lobes)'],
        [sg_exp, sg_gau, sg_mul],
        ['steelblue', 'darkorange', 'forestgreen']):
    ax2 = ax.twinx()
    ax.plot(vmem_vals, step, color='black', linewidth=2, linestyle='--', label='Step (forward)')
    ax.axvline(threshold, color='red', linewidth=1, linestyle=':', alpha=0.6)
    ax.set_ylim(-0.3, 1.4); ax.set_ylabel('spike (0 or 1)', fontsize=10)
    ax2.plot(vmem_vals, sg, color=color, linewidth=2.5, label='Surrogate grad')
    ax2.axhline(0, color='gray', linewidth=0.8, alpha=0.4)
    ax2.set_ylabel('gradient magnitude', fontsize=10, color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    ax.set_title(name, fontsize=11, fontweight='bold'); ax.set_xlabel('Vmem', fontsize=10)
    ax.text(threshold + 0.05, 0.05, 'θ', color='red', fontsize=12)
    lines1, l1 = ax.get_legend_handles_labels()
    lines2, l2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, l1 + l2, fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

### BPTT: backpropagation through time

During training, the forward pass unrolls the neuron across all 20 timesteps. The backward pass flows gradients back through the same graph — this is **BPTT**.

With 20 timesteps, the gradient at t=0 depends on what happened at t=1, 2, ..., 19. This means early timesteps (the beginning of the gesture) do receive gradient signal, but it is weaker than for late timesteps. The surrogate gradient is what makes this signal non-zero at all.

In [ ]:
# BPTT illustration over 20 timesteps (gesture-length)
T_bptt = 20
np.random.seed(3)
toy_inputs = np.clip(np.random.normal(0.12, 0.06, T_bptt), 0.01, 0.4)
threshold_val, beta_val = 1.0, 4.0

vmem_trace, spikes_trace = iaf_neuron_sim(toy_inputs, threshold=threshold_val)
vmem_display = vmem_trace.copy()
for t in np.where(spikes_trace > 0)[0]:
    vmem_display[t] = vmem_trace[t] + spikes_trace[t] * threshold_val
sg_vals = np.exp(-beta_val * np.abs(vmem_display - threshold_val))

fig = plt.figure(figsize=(15, 9))
gs = gridspec.GridSpec(3, 1, hspace=0.55)
ax_input, ax_vmem, ax_grad = [fig.add_subplot(gs[i]) for i in range(3)]
fig.suptitle('BPTT across 20 timesteps — a full gesture duration', fontsize=13, fontweight='bold')
ts = np.arange(T_bptt)

ax_input.bar(ts, toy_inputs, color='steelblue', alpha=0.8, width=0.7)
ax_input.set_ylabel('Input current', fontsize=10)
ax_input.set_title('① Input current — sparse motion signal at each timestep', fontsize=10)
ax_input.set_xticks(ts[::2]); ax_input.grid(axis='y', alpha=0.3)

ax_vmem.plot(ts, vmem_display, color='darkorange', linewidth=2, marker='o', markersize=4)
ax_vmem.axhline(threshold_val, color='red', linestyle='--', linewidth=1.5, label='Threshold θ')

ap_fwd = dict(arrowstyle='->', color='darkorange', lw=1, alpha=0.4)
for t in range(T_bptt - 1):
    ax_vmem.annotate('', xy=(t + 0.85, vmem_display[t + 1]), xytext=(t + 0.15, vmem_display[t]), arrowprops=ap_fwd)
for t in range(T_bptt - 1, 0, -1):
    ax_vmem.annotate('', xy=(t - 0.85, vmem_display[t - 1] + 0.07), xytext=(t - 0.15, vmem_display[t] + 0.07),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.2, alpha=float(0.2 + 0.6 * sg_vals[t])))

for t in np.where(spikes_trace > 0)[0]:
    ax_vmem.annotate('', xy=(t, vmem_trace[t]), xytext=(t, vmem_display[t]),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, connectionstyle='arc3,rad=0.4'))
    lbl = 'spike' if spikes_trace[t] == 1 else f'{int(spikes_trace[t])}×'
    ax_vmem.text(t, vmem_display[t] + 0.08, lbl, ha='center', fontsize=8, color='red', fontweight='bold')

ax_vmem.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in
               [('darkorange', 'Forward (Vmem chain)'), ('purple', '← Backward (BPTT)'),
                ('red', 'Threshold / spike')]], fontsize=9, loc='upper left')
ax_vmem.set_ylabel('Vmem', fontsize=10)
ax_vmem.set_title('② Vmem — forward chain (orange) and backward gradient flow (purple)', fontsize=10)
ax_vmem.set_xticks(ts[::2]); ax_vmem.set_ylim(-0.1, 1.7); ax_vmem.grid(axis='y', alpha=0.3)

ax_grad.bar(ts, sg_vals, color=['red' if s > 0 else 'steelblue' for s in spikes_trace], alpha=0.8, width=0.7)
ax_grad.set_ylabel('Surrogate gradient', fontsize=10)
ax_grad.set_title('③ Gradient signal per timestep — red = spike occurred', fontsize=10)
ax_grad.set_xticks(ts[::2]); ax_grad.set_ylim(0, 1.1); ax_grad.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Training Loop

After the forward pass, output spikes have shape `(B×T, 11)`. We sum across T to get one prediction per sample:

```
output spikes:   (B×T, 11)  — binary
                   ↓  reshape to (B, T, 11)  →  sum over T
spike counts:    (B, 11)    — total spikes per gesture class per sample
                   ↓  cross-entropy
loss:            scalar
```

The class with the most spikes wins — the network learns which output neuron to drive above threshold most often.

In [ ]:
from tonic.collation import PadTensors

train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=PadTensors(batch_first=True),
)
test_loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    collate_fn=PadTensors(batch_first=True),
)

x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: {tuple(x_batch.shape)}   (B, T, C, H, W)")
print(f"Labels:      {tuple(y_batch.shape)}")
print(f"Gesture:     {GESTURE_CLASSES[y_batch[0].item()]}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

model = build_snn(BATCH_SIZE, NUM_TIMESTEPS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.float().to(device), y.long().to(device)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        spike_counts = model(x).reshape(B, T, -1).sum(dim=1)
        loss = loss_fn(spike_counts, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        sinabs.reset_states(model)
        total_loss += loss.item()
        correct += (spike_counts.argmax(dim=1) == y).sum().item()
        total += B
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.float().to(device), y.long().to(device)
        B, T, C, H, W = x.shape
        spike_counts = model(x.reshape(B * T, C, H, W)).reshape(B, T, -1).sum(dim=1)
        loss = loss_fn(spike_counts, y)
        sinabs.reset_states(model)
        total_loss += loss.item()
        correct += (spike_counts.argmax(dim=1) == y).sum().item()
        total += B
    return total_loss / len(loader), correct / total

In [ ]:
NUM_EPOCHS = 10  # DVS Gesture is small (1077 samples) — more epochs are affordable

train_losses, train_accs = [], []
test_losses,  test_accs  = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    te_loss, te_acc = evaluate(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss);  train_accs.append(tr_acc)
    test_losses.append(te_loss);   test_accs.append(te_acc)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}  "
          f"train loss: {tr_loss:.4f}  train acc: {tr_acc:.3f}  "
          f"test loss: {te_loss:.4f}  test acc: {te_acc:.3f}")

---
## 7. Results

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Training results — SNN on DVS Gesture', fontsize=13, fontweight='bold')

ax1.plot(epochs, train_losses, 'o-', label='Train', color='steelblue')
ax1.plot(epochs, test_losses,  's--', label='Test',  color='darkorange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Cross-entropy loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, train_accs, 'o-', label='Train', color='steelblue')
ax2.plot(epochs, test_accs,  's--', label='Test',  color='darkorange')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Classification accuracy')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# Use a random batch so the visualisation is representative (not always the same first batch)
rand_loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=PadTensors(batch_first=True),
)
model.eval()
x_vis, y_vis = next(iter(rand_loader))
with torch.no_grad():
    out = model(x_vis.float().to(device).reshape(BATCH_SIZE * NUM_TIMESTEPS, 2, 128, 128)).cpu()
sinabs.reset_states(model)
spike_counts_vis = out.reshape(BATCH_SIZE, NUM_TIMESTEPS, 11).sum(dim=1).numpy()

overall_test_acc = test_accs[-1]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle(
    f'Spike counts per output neuron — 6 random test samples\n'
    f'Overall test accuracy (epoch {NUM_EPOCHS}): {overall_test_acc:.1%}',
    fontsize=12, fontweight='bold'
)
for i, ax in enumerate(axes.flat):
    counts = spike_counts_vis[i]
    true_lbl, pred_lbl = y_vis[i].item(), counts.argmax()
    correct = pred_lbl == true_lbl
    bar_colors = ['green' if c == true_lbl else ('red' if c == pred_lbl and not correct else 'steelblue')
                  for c in range(11)]
    ax.bar(range(11), counts, color=bar_colors, alpha=0.8)
    ax.set_xticks(range(11))
    ax.set_xticklabels([g[:8] for g in GESTURE_CLASSES], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Spike count', fontsize=9)
    result = '✓' if correct else '✗'
    ax.set_title(f'{result} True: {GESTURE_CLASSES[true_lbl][:12]}\n'
                 f'   Pred: {GESTURE_CLASSES[pred_lbl][:12]}', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


---
## Summary

| Concept | Key point |
|---|---|
| Why DVS Gesture | Gesture identity is in the motion over time — a single frame is ambiguous |
| Spiking neuron | Accumulates evidence in Vmem; fires when charge crosses threshold |
| IAFSqueeze | Squeezes B×T → flat batch for Conv layers; loops over T internally; keeps Vmem per sample |
| Surrogate gradient | Replaces zero derivative of step function in backward pass |
| Loss | Spike counts summed over all 20 timesteps → cross-entropy |
| BPTT | Gradients flow back through all 20 timesteps; surrogate makes this possible |

**DVS Gesture vs N-MNIST:**  
N-MNIST works with T=1 (a single frame still shows a digit). DVS Gesture does not — the gesture only exists across the full sequence, compatible with SNNs and temporal integration.

**Next steps:**
- Increase `NUM_TIMESTEPS` — does accuracy improve?
- Try the exercises in `snn_exercises.ipynb` with this dataset
- Explore the [sinabs documentation](https://sinabs.readthedocs.io) 